# 11 · Pick the right tool, then check it's allowed

## The problem

An agent has a toolbox: search the docs, look up an order, issue a refund, file a bug, send an email. For every step
it has to pick the tool that fits, and sometimes **no tool** fits at all. If the same model both picks and runs the
tool, one misread request is enough to refund a payment when the user only asked a question, or to let a read-only
support bot move money.

## Why Jev

Choosing a tool is a classification over the tools' **purposes**, made on every agent step. Jev's `Choice` does that
as a typed, **calibrated** judgment in about 100 ms for a fraction of a cent, and returns a probability for **every**
tool, so you can see when it is torn between two.

Just as important is what Jev does **not** do: it only *proposes*. Permissions and required arguments are checked in
plain code, which can't be talked out of a rule.

> **Model selects, code authorizes.**

## What we'll build

```
request + caller role ──► Jev: Choice over tool purposes ──► code checks, in order ──► EXECUTE / ANSWER / ASK / DENIED
                           (includes an explicit "none")      1. confident enough?        (ASK)
                                                              2. "none"?                  (ANSWER)
                                                              3. role allowed?            (DENIED)
                                                              4. required args present?   (ASK)
```

## What you'll learn
- How a `Choice` works: `.choice`, `.confidence`, and `.probabilities` over every label.
- Why every tool selector needs an explicit **escape option** (`none`), and how to test it.
- The *select → authorize → validate* pipeline, with permissions and arguments in code.
- Why tool **descriptions** are the model's whole world, and how a wording change fixed a real mistake.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #2.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads `TYPESAFE_API_KEY` from the `.env` file at the repo root,
- creates the Jev client `jev`,
- imports `Choice`, the only Jev question type this notebook needs,
- imports `re` (we use regexes to find required arguments) and defines `bar()`, a small text bar for probabilities.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The toolbox, the permissions and the required arguments

Four pieces, all plain data:

- **`TOOLS`**: tool name → a one-line description of its **purpose**. This dict becomes the options of the `Choice`,
  so these descriptions are all Jev ever knows about each tool. Note the explicit **`none`** option.
- **`PERMISSIONS`**: which roles may call which tools. A `viewer` can only read; an `agent` can also file tickets and
  send email; only an `admin` can refund.
- **`REQUIRED_ARGS`**: for tools that need an argument, a regex that must find it in the request (an order number like
  `#88231`, an email address).
- **`TOOL_QUESTION`**: the single Jev question. **`Choice`** takes `criteria`, a dict of `label → description`, and
  returns `.choice` (the winning label), `.confidence` (how sure Jev is) and `.probabilities` (a dict with one
  probability per label). The `state` is just the request string.

This cell only **defines** them. Nothing is sent to Jev yet.

In [2]:
TOOLS = {
    "search_docs":    "Answer a how-to question from the help-center articles",
    "query_orders":   "Read-only lookup: where an order is, or what it contained",
    "refund_payment": "Give the customer their money back for an order (issue a refund)",
    "create_ticket":  "Report a bug so an engineer investigates it",
    "send_email":     "Send an email to a specific address",
    "none":           "No tool fits: answer directly, or ask a clarifying question",
}

PERMISSIONS = {   # which roles may call which tools
    "viewer": {"search_docs", "query_orders"},
    "agent":  {"search_docs", "query_orders", "create_ticket", "send_email"},
    "admin":  set(TOOLS),
}

REQUIRED_ARGS = {   # argument name -> regex that finds it in the request
    "query_orders":   {"order_id": r"#\d{4,}"},
    "refund_payment": {"order_id": r"#\d{4,}"},
    "send_email":     {"recipient": r"[\w.+-]+@[\w-]+\.[\w.]+"},
}

TOOL_QUESTION = {"tool": Choice(
    instructions="Which tool's purpose fits what the user is asking to be DONE? Choose 'none' if no tool fits.",
    criteria=TOOLS,
)}

## Step 2 · Look at the full distribution

Before building any policy, look at what Jev actually returns. We print the top three labels from `.probabilities`
for a request that clearly needs a tool, and one that fits **no** tool. The second should land on `none`.

> **Workshop note:** the first draft described `query_orders` as *"look up the status or details of an order"*, and *"Refund order #88231"* went there instead of to `refund_payment`, because both mention *an order*. Making the descriptions about the **action** (read-only lookup vs. give money back) fixed it. Tool descriptions are the model's whole world, so test look-alikes.

In [3]:
for req in ["Can you refund order #88231? It arrived broken.", "What's the weather in Paris tomorrow?"]:
    c = jev.system_one(state=req, questions=TOOL_QUESTION).choices["tool"]
    print(req)
    for name, p in sorted(c.probabilities.items(), key=lambda kv: -kv[1])[:3]:
        print(f"   {name:<15} {bar(p)} {p:.2f}")
    print()

Can you refund order #88231? It arrived broken.
   refund_payment  ████████████████████ 0.99
   none            ···················· 0.01
   search_docs     ···················· 0.00

What's the weather in Paris tomorrow?
   none            ████████████████████ 1.00
   search_docs     ···················· 0.00
   query_orders    ···················· 0.00



**What to notice**
- The refund request puts almost all its probability on `refund_payment`, not on the look-alike `query_orders`.
- The weather question goes to **`none`**: no tool fits, and Jev says so instead of forcing the nearest tool. Without
  a `none` option it would have had to pick one, probably `search_docs`.
- `.probabilities` covers every tool. When two tools split the probability, that is a signal to ask, not to act.

## Step 3 · Select → authorize → validate arguments

`plan_step()` is the whole pipeline. Jev's pick is checked by code, **in this order**, and the first check that fails
decides the outcome:

| # | Check | Outcome if it fails | Why this order |
|---|---|---|---|
| 1 | `confidence >= MIN_CONF` | **ASK**: unsure which tool fits | don't authorize a guess |
| 2 | tool is not `none` | **ANSWER**: reply directly, no tool | nothing to authorize |
| 3 | tool is in `PERMISSIONS[role]` | **DENIED** | permissions before arguments: never ask a viewer for an order number to a refund they can't do |
| 4 | every `REQUIRED_ARGS` regex matches | **ASK**: missing argument | don't let a tool guess an order id |
| - | all passed | **EXECUTE** with the extracted arguments | |

Note that steps 3 and 4 involve **no model at all**. A cleverly worded request can't talk its way past a set
membership test or a regex.

In [4]:
MIN_CONF = 0.60   # below this, Jev isn't sure which tool fits: ask the user


def plan_step(request, role, client):
    if "TOOL_QUESTION" not in globals():
        raise RuntimeError("TOOL_QUESTION is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    c = client.system_one(state=request, questions=TOOL_QUESTION).choices["tool"]
    tool, conf = c.choice, c.confidence

    if conf < MIN_CONF:
        return {"tool": tool, "conf": conf, "status": "ASK", "detail": "unsure which tool fits; ask the user"}
    if tool == "none":
        return {"tool": tool, "conf": conf, "status": "ANSWER", "detail": "no tool needed / no tool fits"}
    if tool not in PERMISSIONS[role]:
        return {"tool": tool, "conf": conf, "status": "DENIED", "detail": f"role '{role}' may not call {tool}"}

    args, missing = {}, []
    for arg, pattern in REQUIRED_ARGS.get(tool, {}).items():
        m = re.search(pattern, request)
        if m:
            args[arg] = m.group(0)
        else:
            missing.append(arg)
    if missing:
        return {"tool": tool, "conf": conf, "status": "ASK", "detail": f"missing argument(s): {', '.join(missing)}"}
    return {"tool": tool, "conf": conf, "status": "EXECUTE", "detail": f"{tool}({args})"}

## Step 4 · Run it

Eight requests, chosen to hit every outcome. The same refund request appears twice with different roles: the tool
choice is identical, only the permission check differs. This cell only defines the list.

In [5]:
REQUESTS = [
    ("How do I change my password?", "viewer"),
    ("Where is my order #10442?", "viewer"),
    ("Refund order #88231, it arrived broken.", "agent"),
    ("Refund order #88231, it arrived broken.", "admin"),
    ("Please refund my last order.", "admin"),
    ("Email the invoice to jane@acme.example", "agent"),
    ("The checkout button crashes on Safari every time.", "agent"),
    ("What's the weather in Paris tomorrow?", "agent"),
]
print(len(REQUESTS), "requests")

8 requests


Now plan each step. Each line shows the outcome, the caller's role, the request, Jev's tool pick with its confidence,
and the detail: the extracted arguments for EXECUTE, or the reason it stopped.

In [6]:
for req, role in REQUESTS:
    d = plan_step(req, role, jev)
    print(f"[{d['status']:^7}] {role:<6} {req[:45]:<45} -> {d['tool']} ({d['conf']:.2f}): {d['detail']}")

[EXECUTE] viewer How do I change my password?                  -> search_docs (0.68): search_docs({})


[EXECUTE] viewer Where is my order #10442?                     -> query_orders (1.00): query_orders({'order_id': '#10442'})


[DENIED ] agent  Refund order #88231, it arrived broken.       -> refund_payment (0.99): role 'agent' may not call refund_payment


[EXECUTE] admin  Refund order #88231, it arrived broken.       -> refund_payment (0.99): refund_payment({'order_id': '#88231'})


[  ASK  ] admin  Please refund my last order.                  -> refund_payment (0.98): missing argument(s): order_id


[EXECUTE] agent  Email the invoice to jane@acme.example        -> send_email (1.00): send_email({'recipient': 'jane@acme.example'})


[EXECUTE] agent  The checkout button crashes on Safari every t -> create_ticket (0.98): create_ticket({})


[ANSWER ] agent  What's the weather in Paris tomorrow?         -> none (1.00): no tool needed / no tool fits


**What to notice**
- **Same request, different role, different outcome:** the refund is `DENIED` for an `agent` and `EXECUTE`d for an
  `admin`. Jev's choice didn't change; the code's permission check did the work.
- *"Please refund my last order"* picks the right tool with high confidence but comes back **ASK**: there's no order
  number, and the code refuses to let the tool guess one.
- The order lookup and the email both execute with arguments **extracted by regex** (`#10442`, `jane@acme.example`),
  not invented by a model.
- The password question has the **lowest confidence** of the batch: some probability goes to `none`, whose description
  includes *"answer directly"*. It still clears `MIN_CONF`, but it is the request to watch if you raise the threshold.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `plan_step()` with `ipywidgets`,
and an HTML card per decision. The selection and authorization logic above is unchanged.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`tool_html()` renders one decision card: status badge, chosen tool, confidence, role, and the detail line. Below we
show the three refund-related requests from Step 4 (denied, executed, and missing an argument).

In [8]:
ST_COLOR = {"EXECUTE": SAGE, "ANSWER": SLATE, "ASK": AMBER, "DENIED": RED}


def tool_html(request, role):
    role = role if role in PERMISSIONS else "agent"
    d = plan_step(request, role, jev)
    return box(f'<div style="{CARD}border-left:5px solid {ST_COLOR[d["status"]]};">'
               f'{badge(d["status"], ST_COLOR[d["status"]])} <code>{d["tool"]}</code> '
               f'<span style="color:{SLATE};font-size:12px;">confidence {d["conf"]:.2f} · role {role}</span>'
               f'<div style="margin-top:6px;">{request}</div>'
               f'<div style="color:{SLATE};font-size:12px;margin-top:4px;">{d["detail"]}</div></div>')


display(HTML(box("".join(tool_html(q, r) for q, r in REQUESTS[2:5]))))

## Step 6 · Try it yourself

Edit the request and the role (`viewer`, `agent` or `admin`), then click **Plan step** (needs a live kernel). Good ones
to try:
- the default, `Refund order #55120 please` as `agent` → **DENIED**; change the role to `admin` → **EXECUTE**
- `I want my money back` as `admin` → **ASK** (right tool, no order number)
- `Send the invoice to my accountant` as `agent` → `send_email`, then **ASK** for the missing recipient address
- `Hmm, something's off with my account` → **ANSWER** via `none`: too vague for any tool, so reply or ask a question

In [9]:
form_app([("request", "User request", "Refund order #55120 please", 1),
          ("role", "Caller role (viewer / agent / admin)", "agent", 1)], tool_html, "Plan step")

Output()

## Recap and exercises

**Key ideas**
1. **Model selects, code authorizes.** Jev proposes a tool; permissions and arguments are plain code it can't bypass.
2. Always include an **escape option** (`none`) and test requests that fit nothing.
3. Tool **descriptions** are the model's whole world. Describe the *action* (read-only lookup vs. give money back), and test look-alikes.
4. Use `.confidence` and `.probabilities`: a split between two tools is a reason to ask, not to act.

**Exercises**
- Raise `MIN_CONF` to 0.8 and re-run Step 4. Which request turns into ASK?
- Add a `cancel_subscription` tool (and give it to `admin` only). Does *"Please refund my last order"* still pick `refund_payment`?
- Remove the `none` option from `TOOLS` and re-run Step 2. Where does the weather question go now?
- Add a `REQUIRED_ARGS` entry for `create_ticket` (e.g. a browser name) and see which bug reports now come back ASK.